# SAE Activation Length Analysis

Find the SAE latents with the largest average activation length, then generate descriptions for them.

In [ ]:
import os
import sys

# Run from the sae-analysis root so relative paths (./data/...) resolve correctly
ROOT = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import json
import numpy as np
import torch
import matplotlib.pyplot as plt

print("Working directory:", os.getcwd())

## Config

In [ ]:
MODEL_NAME = "meta-llama/Llama-3.1-8B"   # subdirectory under ./data/
SAE_NAME   = "Llama-Scope"
LAYER      = 30
TOP_N      = 20    # how many top features to inspect

SAE_PATH   = None  # set to the .pt / .safetensors path of the SAE weights (only needed for description generation)
MODEL_PATH = None  # set to local model dir, or None to use MODEL_NAME from HuggingFace
DEVICE     = "cuda:0"
DTYPE      = "float16"
NORMALIZE_ACTS = False

## 1. Load Activation Length Stats

In [ ]:
sae_dir    = os.path.join("./data", MODEL_NAME, SAE_NAME, f"layer-{LAYER}")
stats_path = os.path.join(sae_dir, "attribute", "activation_length", "stats.pt")

stats  = torch.load(stats_path, map_location="cpu")
avg_len = stats["avg"].numpy()   # shape: (n_features,)
max_len = stats["max"].numpy()
min_len = stats["min"].numpy()

n_features = len(avg_len)
valid_mask = np.isfinite(avg_len) & (avg_len > 0)
print(f"Total features  : {n_features}")
print(f"Ever-activated  : {valid_mask.sum()}")
print(f"Global avg      : {avg_len[valid_mask].mean():.3f} tokens")

## 2. Top-N Features by Average Activation Length

In [ ]:
valid_idx   = np.where(valid_mask)[0]
top_local   = np.argsort(avg_len[valid_idx])[-TOP_N:][::-1]
top_ids     = valid_idx[top_local]          # feature IDs
top_avg     = avg_len[top_ids]
top_max     = max_len[top_ids]
top_min     = min_len[top_ids]

print(f"{'Rank':<6} {'Feature ID':<12} {'Avg Length':>12} {'Max Length':>12} {'Min Length':>12}")
print("-" * 58)
for rank, (fid, al, mx, mn) in enumerate(zip(top_ids, top_avg, top_max, top_min), 1):
    print(f"{rank:<6} {fid:<12} {al:>12.2f} {mx:>12.2f} {mn:>12.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, TOP_N))[::-1]
ax.barh(range(TOP_N), top_avg[::-1], color=colors)
ax.set_yticks(range(TOP_N))
ax.set_yticklabels([f"Feature {fid}" for fid in top_ids[::-1]], fontsize=9)
ax.set_xlabel("Average Activation Length (tokens)")
ax.set_title(
    f"Top {TOP_N} SAE Features by Average Activation Length\n"
    f"{MODEL_NAME} / {SAE_NAME} / Layer {LAYER}"
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print("Feature IDs (largest first):", top_ids.tolist())

## 3. Generate Descriptions

This section loads the model and SAE, then calls `describe_features()` from `utils/description.py`  
to generate per-feature description files under `<sae_dir>/description/feature_<id>/`.

> **Skip this section** if you only need the ranked list above — it does not require a GPU or model weights.

In [ ]:
from utils.hf_models.model_factory import construct_model_base
from utils.utils import model_alias_to_model_name
from utils.sae.sae_base import SAEBase
from utils.description import describe_features

model_path = MODEL_PATH if MODEL_PATH is not None else MODEL_NAME
model_base = construct_model_base(model_path, MODEL_NAME)

sae_base = SAEBase(
    SAE_PATH,
    device=DEVICE,
    normalize_acts=NORMALIZE_ACTS,
    top_k=None,   # uses sae_dim // 100 by default
    layer=LAYER,
    name=SAE_NAME,
    dtype=DTYPE,
)

In [ ]:
feature_dirs = describe_features(
    model_base=model_base,
    sae_base=sae_base,
    feature_ids=top_ids.tolist(),
    text_k=40,
    context_window_forward=20,
    context_window_backward=20,
    top_grad_k=5,
    chunk_samples=1024,
)

## 4. Display Generated Descriptions

In [ ]:
def show_feature(feature_idx: int, fdir: str, n_contexts: int = 3):
    """Pretty-print the generated description files for one feature."""
    sep = "=" * 70
    print(f"\n{sep}")
    print(f"  Feature {feature_idx}   (avg length: {avg_len[feature_idx]:.2f} tokens)")
    print(sep)

    feat_path = os.path.join(fdir, "feature.json")
    if os.path.exists(feat_path):
        print("\n[Stats]")
        with open(feat_path) as f:
            print(json.dumps(json.load(f), indent=2))

    entropy_path = os.path.join(fdir, "decoder_entropy.json")
    if os.path.exists(entropy_path):
        print("\n[Decoder top-10 tokens]")
        with open(entropy_path) as f:
            data = json.load(f)
        print(f"  Entropy: {data['decoder_entropy']:.3f}")
        for item in data["top10_decoder_tokens"]:
            print(f"  {item['token']!r:20s} prob={item['prob']:.4f}")

    topk_path = os.path.join(fdir, "topk_text.jsonl")
    if os.path.exists(topk_path):
        print(f"\n[Top {n_contexts} activating contexts]")
        with open(topk_path) as f:
            lines = f.readlines()
        count = 0
        for line in lines[1:]:  # skip header
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                print(f"  [act={entry['activation']:.3f}] {entry['text'][:300]}")
                count += 1
                if count >= n_contexts:
                    break
            except json.JSONDecodeError:
                pass


for fid, fdir in feature_dirs.items():
    show_feature(fid, fdir, n_contexts=3)